In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import corc.utils
import corc.purity
import scipy.cluster.hierarchy
import sklearn.metrics
import numpy as np

In [36]:
dataset = "noisy_circles"
# dataset = "densired8"
X,y,tsne = corc.utils.load_dataset(dataset)
tmms = corc.utils.load_tmms(dataset)
tmm = tmms[0]

In [37]:
single = corc.utils.load_algorithms(dataset, "AgglomerativeClustering")
agglomerative = single[0]
linkage_matrix = np.column_stack(
        [
            agglomerative.children_,
            # agglomerative.distances_,
            np.ones(agglomerative.children_.shape[0]),
            np.ones(agglomerative.children_.shape[0]),
        ]
    ).astype(float)

In [44]:
corc.purity.dendrogram_purity(linkage_matrix, y)

0.5986312684411623

In [43]:
tmm.get_purity(X,y, recompute=True)

1.0

In [21]:
tmm.get_purity(X,y)

1.0

In [9]:
adj = -tmm.adjacency_.copy()

In [10]:
condensed_linearized = scipy.spatial.distance.squareform(adj, checks=False)

In [33]:
y_pred = tmm.mixture_model.predict(X)

In [37]:
np.bincount(y_pred)

array([333, 435, 409, 494, 389, 308, 384, 613, 267, 445, 433, 385, 619,
       366, 408, 332, 377, 265, 354, 405, 509, 372, 430, 324, 344])

In [39]:
tmm.mixture_model.get_counts(X)

Array([333, 435, 409, 494, 389, 308, 384, 613, 267, 445, 433, 385, 619,
       366, 408, 332, 377, 265, 354, 405, 509, 372, 430, 324, 344],      dtype=int32)

In [14]:
Z = scipy.cluster.hierarchy.linkage(condensed_linearized, method='single')

In [23]:
Z[:,2] += 3

In [24]:
raw_labels = scipy.cluster.hierarchy.fcluster(Z, 6, criterion='maxclust')

In [25]:
raw_labels

array([2, 5, 3, 3, 2, 2, 2, 4, 6, 3, 3, 1, 4, 2, 3, 3, 5, 6, 5, 3, 3, 3,
       3, 1, 2], dtype=int32)

In [17]:
thresh, assignments = tmm.get_thresholds_and_cluster_numbers()

In [31]:
n= 8
raw_labels = scipy.cluster.hierarchy.fcluster(Z, n, criterion='maxclust')
sklearn.metrics.adjusted_rand_score(assignments[n], raw_labels)

1.0